# google_cloud_storage smoke test

Smoke test for `google-cloud-storage`. Import name is `google.cloud.storage`
(NOT `google_cloud_storage`). All examples are client-side only — anonymous
client, URI parsing, URL building, retry policies — no GCS requests are made.

In [ ]:
import pkgutil, importlib
import google.cloud.storage as _pkg_under_test  # import name for the google-cloud-storage dist

failed = []
for module_info in pkgutil.walk_packages(_pkg_under_test.__path__, prefix=f"{_pkg_under_test.__name__}."):
    try:
        importlib.import_module(module_info.name)
    except Exception as e:
        failed.append((module_info.name, type(e).__name__, str(e)))

if failed:
    for name, etype, msg in failed:
        print(f"FAIL {name}: {etype}: {msg}")
    raise AssertionError(f"{len(failed)} submodule(s) of {_pkg_under_test.__name__} failed to import")
print(f"OK — all submodules of {_pkg_under_test.__name__} imported cleanly")

In [ ]:
# Example 1: Anonymous client construction (offline — no credentials lookup).
from google.cloud import storage

client = storage.Client.create_anonymous_client()
print(f"client: {type(client).__name__}")
print(f"project: {client.project}")
print(f"credentials: {type(client._credentials).__name__}")
assert client.project is None
assert type(client._credentials).__name__ == "AnonymousCredentials"
print(f"google-cloud-storage version: {storage.__version__}")

In [ ]:
# Example 2: Parse a gs:// URI into Bucket/Blob (pure client-side).
from google.cloud import storage
from google.cloud.storage import Blob

client = storage.Client.create_anonymous_client()
blob = Blob.from_uri("gs://my-bucket/path/to/object.txt", client=client)
print(f"bucket: {blob.bucket.name}")
print(f"name:   {blob.name}")
print(f"path:   {blob.path}")
assert blob.bucket.name == "my-bucket"
assert blob.name == "path/to/object.txt"
assert blob.path == "/b/my-bucket/o/path%2Fto%2Fobject.txt"

In [ ]:
# Example 3: Bucket/Blob factories + public URL building (URL-encoding invariant).
from google.cloud import storage

client = storage.Client.create_anonymous_client()
bucket = client.bucket("demo-bucket")
blob = bucket.blob("folder/file with space.txt")

print(f"bucket.path: {bucket.path}")
print(f"public_url:  {blob.public_url}")
assert bucket.path == "/b/demo-bucket"
assert blob.public_url == (
    "https://storage.googleapis.com/demo-bucket/folder/file%20with%20space.txt"
)
print("client-side URL building is correct")

In [ ]:
# Example 4: Retry policy objects — DEFAULT_RETRY + conditional retry wrappers.
from google.api_core.retry import Retry
from google.cloud.storage.retry import (
    DEFAULT_RETRY,
    DEFAULT_RETRY_IF_GENERATION_SPECIFIED,
)

print(f"DEFAULT_RETRY: {DEFAULT_RETRY}")
print(f"is api_core Retry: {isinstance(DEFAULT_RETRY, Retry)}")
custom = DEFAULT_RETRY.with_timeout(60.0).with_delay(initial=0.5, maximum=8.0)
print(f"customized retry timeout: {custom._timeout}")
assert isinstance(DEFAULT_RETRY, Retry)
assert DEFAULT_RETRY_IF_GENERATION_SPECIFIED is not None

In [ ]:
# Example 5: google-crc32c (a C extension) backs storage's
# download/upload integrity checks.
import google_crc32c

value = google_crc32c.value(b"The quick brown fox jumps over the lazy dog")
print(f"crc32c: {value:#010x}")
print(f"implementation: {google_crc32c.implementation}")
assert value == 0x22620404  # RFC 3720 / well-known CRC32C test vector
checksum = google_crc32c.Checksum(b"The quick brown fox ")
checksum.update(b"jumps over the lazy dog")
assert int.from_bytes(checksum.digest(), "big") == value
print("streaming Checksum matches one-shot value")